# 17_writing_systems: What Survives Script Change?

**Lab report:** Invariant Specification Demystifies Translation

Notebook 13 examined translation:

```text
Language A ↔ Language B
```

Notebook 17 examines script change:

```text
Language
  ├─ Script A
  ├─ Script B
  └─ Script C
```

The core question is:

> If meaning and language stay fixed, what changes when only the writing system changes?

This is a cleaner invariant test than translation, because script can vary while grammar, lexicon, and meaning may remain stable.

## 1. Setup

Clone or reuse the upstream GLOSSOPETRAE repository.

In [ ]:
from pathlib import Path
import os
import re
import json
import csv
import subprocess
from collections import Counter, defaultdict

ROOT = Path.cwd()
UPSTREAM_URL = "https://github.com/elder-plinius/GLOSSOPETRAE.git"
UPSTREAM_DIR = ROOT / "GLOSSOPETRAE"

if not UPSTREAM_DIR.exists():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
else:
    print("Upstream repo already present:", UPSTREAM_DIR)

FIGURES = ROOT / "figures"
DATA = ROOT / "data"
FIGURES.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

print("Repository exists:", UPSTREAM_DIR.exists())

## 2. Search for writing-system artifacts

Locate code, configs, examples, and docs related to scripts, glyphs, orthography, SVG rendering, and visual writing systems.

In [ ]:
script_terms = [
    "script",
    "glyph",
    "svg",
    "orthography",
    "writing",
    "render",
    "alphabet",
    "character",
    "letter",
    "font",
    "calligraphy",
    "stroke",
    "grapheme",
    "unicode",
]

TEXT_SUFFIXES = {
    ".py", ".md", ".txt", ".yaml", ".yml", ".json", ".toml",
    ".js", ".ts", ".html", ".css", ".csv", ".tsv", ".svg"
}

path_hits = []
content_hits = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir():
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower_path = str(rel).lower()

    hits = [t for t in script_terms if t in lower_path]
    if hits:
        path_hits.append({
            "path": str(rel),
            "suffix": path.suffix,
            "hits": ", ".join(hits),
            "size_bytes": path.stat().st_size if path.exists() else None,
        })

    if path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue

    lower = text.lower()
    for term in script_terms:
        count = lower.count(term)
        if count:
            content_hits.append({
                "path": str(rel),
                "term": term,
                "count": count,
            })

len(path_hits), len(content_hits)

In [ ]:
try:
    import pandas as pd

    path_df = pd.DataFrame(path_hits).sort_values(["hits", "path"])
    content_df = pd.DataFrame(content_hits)

    display(path_df.head(80))

    term_summary = (
        content_df
        .groupby("term", as_index=False)["count"]
        .sum()
        .sort_values("count", ascending=False)
    )
    display(term_summary)

    path_df.to_csv(DATA / "17_script_path_hits.csv", index=False)
    term_summary.to_csv(DATA / "17_script_term_summary.csv", index=False)
except Exception:
    print("Path hits:")
    for row in path_hits[:80]:
        print(row)
    print("\\nTerm counts:")
    print(Counter([r["term"] for r in content_hits]))

## 3. Extract writing-system snippets

Look for text near high-value terms. The goal is to determine whether scripts are:

1. specified directly,
2. generated from a language specification,
3. rendered as visual glyphs,
4. linked to phonology or orthography,
5. independent decorative outputs.

In [ ]:
def snippets_for(term, max_snippets=20, radius=180):
    rows = []
    for path in UPSTREAM_DIR.rglob("*"):
        if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
            continue

        try:
            text = path.read_text(errors="ignore")
        except Exception:
            continue

        lower = text.lower()
        start_search = 0
        while len(rows) < max_snippets:
            idx = lower.find(term.lower(), start_search)
            if idx < 0:
                break

            start = max(0, idx - radius)
            end = min(len(text), idx + len(term) + radius)
            snippet = text[start:end].replace("\\n", " ")
            rows.append({
                "term": term,
                "path": str(path.relative_to(UPSTREAM_DIR)),
                "snippet": snippet,
            })
            start_search = idx + len(term)

        if len(rows) >= max_snippets:
            break

    return rows

snippet_terms = ["script", "glyph", "svg", "orthography", "writing", "alphabet", "character"]
snippets = []
for term in snippet_terms:
    snippets.extend(snippets_for(term, max_snippets=12))

try:
    import pandas as pd
    snippets_df = pd.DataFrame(snippets)
    display(snippets_df)
    snippets_df.to_csv(DATA / "17_script_snippets.csv", index=False)
except Exception:
    for s in snippets:
        print("\\n---", s["term"], s["path"])
        print(s["snippet"])

## 4. Candidate script files

Rank likely writing-system files by path and content evidence.

In [ ]:
candidate_script_files = []

for path in UPSTREAM_DIR.rglob("*"):
    if ".git" in path.parts or path.is_dir() or path.suffix.lower() not in TEXT_SUFFIXES:
        continue

    rel = path.relative_to(UPSTREAM_DIR)
    lower = str(rel).lower()

    path_score = sum(1 for term in script_terms if term in lower)

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        text = ""

    content_score = sum(text.lower().count(term) for term in script_terms)

    if path_score or content_score:
        candidate_script_files.append({
            "path": str(rel),
            "suffix": path.suffix,
            "size_bytes": path.stat().st_size,
            "path_score": path_score,
            "content_score": content_score,
            "total_score": path_score * 10 + content_score,
        })

candidate_script_files = sorted(
    candidate_script_files,
    key=lambda r: (-r["total_score"], r["size_bytes"], r["path"])
)

try:
    import pandas as pd
    script_files_df = pd.DataFrame(candidate_script_files)
    display(script_files_df.head(80))
    script_files_df.to_csv(DATA / "17_candidate_script_files.csv", index=False)
except Exception:
    for row in candidate_script_files[:80]:
        print(row)

## 5. Script-change invariant table

For a same-language / different-script comparison, record what changes and what persists.

The cleanest expected outcome is:

```text
same meaning
same language
same pronunciation
different glyphs
```

In [ ]:
script_change_rows = [
    {
        "property": "meaning / concept",
        "expected_under_script_change": "preserved",
        "observed": "unresolved",
        "notes": "Script should not change intended meaning."
    },
    {
        "property": "lexicon / words",
        "expected_under_script_change": "preserved",
        "observed": "unresolved",
        "notes": "Underlying words may remain the same while written forms change."
    },
    {
        "property": "grammar",
        "expected_under_script_change": "preserved",
        "observed": "unresolved",
        "notes": "Syntax and morphology should be language-level structure."
    },
    {
        "property": "phonology / pronunciation",
        "expected_under_script_change": "preserved or partially represented",
        "observed": "unresolved",
        "notes": "Orthography may encode pronunciation differently."
    },
    {
        "property": "orthography",
        "expected_under_script_change": "changes",
        "observed": "unresolved",
        "notes": "Mapping from sounds/words to visual marks changes."
    },
    {
        "property": "glyph shape",
        "expected_under_script_change": "changes",
        "observed": "unresolved",
        "notes": "The most visible representation-layer change."
    },
    {
        "property": "correspondence",
        "expected_under_script_change": "preserved",
        "observed": "unresolved",
        "notes": "Same language content should correspond across scripts."
    },
]

try:
    import pandas as pd
    script_change_df = pd.DataFrame(script_change_rows)
    display(script_change_df)
    script_change_df.to_csv(DATA / "17_script_change_invariants.csv", index=False)
except Exception:
    for row in script_change_rows:
        print(row)

## 6. SVG and glyph inventory scaffold

If the repo produces SVG glyphs, measure basic visible properties:

- number of SVG files,
- number of path/line/circle elements,
- symbol inventory size,
- file sizes.

This does not prove linguistic structure, but it helps distinguish visual rendering from language-level specification.

In [ ]:
svg_rows = []

for path in UPSTREAM_DIR.rglob("*.svg"):
    if ".git" in path.parts or path.is_dir():
        continue

    try:
        text = path.read_text(errors="ignore")
    except Exception:
        text = ""

    svg_rows.append({
        "path": str(path.relative_to(UPSTREAM_DIR)),
        "size_bytes": path.stat().st_size,
        "path_elements": text.lower().count("<path"),
        "line_elements": text.lower().count("<line"),
        "circle_elements": text.lower().count("<circle"),
        "rect_elements": text.lower().count("<rect"),
        "text_elements": text.lower().count("<text"),
    })

try:
    import pandas as pd
    svg_df = pd.DataFrame(svg_rows)
    display(svg_df.head(80))
    svg_df.to_csv(DATA / "17_svg_inventory.csv", index=False)
except Exception:
    for row in svg_rows[:80]:
        print(row)

print("SVG files found:", len(svg_rows))

## 7. Same-language / multiple-script work area

After locating the runnable script generator, fill this section with the actual command.

Target comparison:

```text
Language X
  ├─ Script A
  ├─ Script B
  └─ Script C
```

Record:

1. same underlying word or phrase,
2. different written forms,
3. whether pronunciation is preserved,
4. whether meaning is preserved,
5. whether correspondence is explicit.

In [ ]:
# TODO: replace with the smallest upstream writing-system command.
#
# Example shape:
#
# result = subprocess.run(
#     ["python", "..."],
#     cwd=UPSTREAM_DIR,
#     text=True,
#     capture_output=True,
#     check=True,
# )
# print(result.stdout[:2000])
#
# Then compare:
#
# same_language = ...
# script_a = ...
# script_b = ...
# same_meaning = ...
# same_pronunciation = ...
# correspondence = ...

## 8. Script correspondence model

Notebook 13 used two languages. Notebook 17 holds language fixed and varies script.

This isolates visual representation from linguistic specification.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axis("off")

nodes = {
    "Specification\\n(constraints)": (0.5, 0.90),
    "Language\\n(grammar + lexicon + sound)": (0.5, 0.68),
    "Correspondence\\n(same language content)": (0.5, 0.48),
    "Script A\\n(glyph set A)": (0.22, 0.25),
    "Script B\\n(glyph set B)": (0.50, 0.25),
    "Script C\\n(glyph set C)": (0.78, 0.25),
    "Visual forms change": (0.5, 0.08),
}

for label, (x, y) in nodes.items():
    ax.text(
        x, y, label,
        ha="center", va="center",
        bbox=dict(boxstyle="round,pad=0.45", linewidth=1.2),
        fontsize=10
    )

edges = [
    ("Specification\\n(constraints)", "Language\\n(grammar + lexicon + sound)"),
    ("Language\\n(grammar + lexicon + sound)", "Correspondence\\n(same language content)"),
    ("Correspondence\\n(same language content)", "Script A\\n(glyph set A)"),
    ("Correspondence\\n(same language content)", "Script B\\n(glyph set B)"),
    ("Correspondence\\n(same language content)", "Script C\\n(glyph set C)"),
    ("Script A\\n(glyph set A)", "Visual forms change"),
    ("Script B\\n(glyph set B)", "Visual forms change"),
    ("Script C\\n(glyph set C)", "Visual forms change"),
]

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2 + 0.04),
        xytext=(x1, y1 - 0.04),
        arrowprops=dict(arrowstyle="->", linewidth=1)
    )

path = FIGURES / "17_script_correspondence.png"
plt.savefig(path, dpi=180, bbox_inches="tight")
print("Saved:", path)
plt.show()

## 9. Hypothesis 3

**Hypothesis 3**

Writing systems encode the same language through distinct visual forms.

Script changes representation.

Language-level correspondences persist.

If this is supported by upstream outputs, then script generation is a useful intermediate test between translation and long-term language evolution.

## 10. Questions for Notebook 23

Notebook 23 should make the transformation stronger:

```text
same specification
      ↓
language over time
      ↓
daughter languages
```

Questions:

| Question | Motivation |
|---|---|
| What survives centuries to millennia of change? | Tests the strongest version of invariant specification |
| Are daughter languages linked by explicit correspondences? | Makes lineage measurable |
| Do sound shifts preserve systematic relationships? | Tests whether change is structured |
| Can descendants differ while retaining family structure? | Connects the repo to language-family generation |